# Controls 96^3: branch ablation, fusion ablation and P baseline

Ablation notebook for the 3-branch model at **96^3** (storage raw 200^3, views from raw 200^3 at 224).
Every variant is **retrained from scratch** on the same split/seed/protocol (no test-time masking), 20
epochs with early stop (`PATIENCE`), best checkpoint by validation AUC, then calibrated test evaluation.
Seed set: `(42, 43, 44)`; results are aggregated as mean +/- std per spec.

| Code | Configuration | `use_3d` | `n_2d` | `view_indices` | `fusion` | `gate_fixed` |
|---|---|---|---|---|---|---|
| P | 3 branches + full CrossGate (main) | True | 2 | (0, 1) | crossgate | False |
| B1 | ResNeXt3D + MaxViT Slab MIP (drop Full AIP) | True | 1 | (0,) | crossgate | False |
| B2 | ResNeXt3D + MaxViT Full AIP (drop Slab MIP) | True | 1 | (1,) | crossgate | False |
| B3 | 2xMaxViT (Slab MIP + Full AIP) + Concat (2D-only) | False | 2 | (0, 1) | concat | - |
| C1 | 3 branches + Concat + classifier | True | 2 | (0, 1) | concat | - |
| C2 | 3 branches + cross-attention, gate fixed = 1 | True | 2 | (0, 1) | crossgate | True |

Views: index 0 = `slab_mip`, index 1 = `aip_full`. B1/B2 keep the projection and CrossGate but the fusion
block only has one 2D token. C2 removes the learned gate parameter (attention + residual add unchanged).

**D1/D2 (Bilateral, 5 epochs)** are run with the reviewed final notebook
(`3d_glaucoma_final_2x2d_3d_crossgate.ipynb`, frozen helper bundle): same pinned warm-start checkpoint
(`.../final_2x2d_3d_crossgate/raw_s42_recovered_20260910/raw_s42/best_weights.pt`), `EPOCHS=5`, `LR=5e-5`,
one run per seed and a new `RUN_GROUP` for each phase.
- D2 (default preset): fine-tune on the Bilateral cache from that checkpoint.
- D1 (control): continue on raw data from the same checkpoint with the equivalent 5-epoch configuration.

`CTRL_SMOKE=1` runs tiny synthetic CPU data for all specs/one seed/one epoch. Real runs need a GPU plus
`WANDB_API_KEY` and `HF_TOKEN`. All artifacts go local + Drive.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SMOKE = os.environ.get('CTRL_SMOKE', '0') == '1'
if not SMOKE and 'google.colab' in sys.modules:
    repo = Path('/content/glaucoma-thesis')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'wandb', 'scipy', 'matplotlib', 'huggingface_hub', 'python-dotenv', 'hf-transfer'], check=True)
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
sys.path.insert(0, str(Path.cwd()))
import json, random, tempfile, time
import numpy as np
import torch
import torch.nn.functional as F
import wandb
import matplotlib
matplotlib.use('Agg')
from scripts import controls_model as cm, final_model as fm, final_training as ft
ft.load_env_file()
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
if SMOKE:
    torch.set_num_threads(1)

## Configuration
`SPECS` is the only source of truth for the variants; model kwargs are recorded in the run config so resume
and comparisons are exact. `EPOCHS=20` with equal `PATIENCE` and best-by-val-AUC for every spec.
`RUN_TARGET` blank selects all specs/seeds; set it to a tag (e.g. `P_s42`) for a single recovery run.
`AUTO_BATCH` probes the largest batch that fits `TARGET_VRAM_GB` (default 60) and keeps the effective batch
via grad accumulation; `RESUME` reuses the saved batch.

In [ ]:
RUN_GROUP = 'crossgate_controls_96'
RUN_TARGET = ''
RESUME = False
EXTEND_EPOCHS = 0
CHECKPOINT_EVERY_STEPS = 10
SPECS = [
    {'code': 'P', 'label': '3 branches + CrossGate', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'crossgate', 'gate_fixed': False},
    {'code': 'B1', 'label': 'ResNeXt3D + slab_mip', 'use_3d': True, 'n_2d': 1, 'view_indices': (0,), 'fusion': 'crossgate', 'gate_fixed': False},
    {'code': 'B2', 'label': 'ResNeXt3D + aip_full', 'use_3d': True, 'n_2d': 1, 'view_indices': (1,), 'fusion': 'crossgate', 'gate_fixed': False},
    {'code': 'B3', 'label': '2D-only + concat', 'use_3d': False, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'concat', 'gate_fixed': False},
    {'code': 'C1', 'label': '3 branches + concat', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'concat', 'gate_fixed': False},
    {'code': 'C2', 'label': '3 branches + attention gate=1', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'crossgate', 'gate_fixed': True},
]
SEEDS = [42] if SMOKE else [42, 43, 44]
STORE_RES = 8 if SMOKE else 200
RES3D = 8 if SMOKE else 96
RES2D = 8 if SMOKE else 224
D_LATENT, ENC2D = 256, 'maxvit_tiny_rw_224'
ENC3D_FEATURES = (32, 64, 128, 192)
EPOCHS, BS, GRAD_ACCUM = (1, 2, 2) if SMOKE else (20, 2, 8)
AUTO_BATCH = not SMOKE
TARGET_VRAM_GB = float(os.environ.get('CTRL_TARGET_VRAM_GB', '60'))
EFFECTIVE_BATCH = 16
NUM_WORKERS = 0 if os.name == 'nt' else 4
LR, WD, PATIENCE = 1e-4, 1e-4, 6
RUN_XAI = False
HF_DATA_REPO = os.environ.get('HF_DATA_REPO', 'tqhuyen/harvard-oct-glaucoma-200')
SPLITS = ('Training', 'Validation', 'Test')
HF_DATA_PATTERNS = [f'{split}_{kind}.npy' for split in SPLITS for kind in ('volumes', 'labels')]
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix='ctrl_smoke_')) if SMOKE else None
DATA_ROOT = SMOKE_ROOT / 'data' if SMOKE else Path('/content/final_data')
LOCAL_ROOT = SMOKE_ROOT / 'runs' if SMOKE else Path('outputs/crossgate_controls_96') / RUN_GROUP
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', '/content/drive'))
DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/MasterBKDN/Thesis'))
DRIVE_DIR = DRIVE_ROOT / 'crossgate_controls_96' / RUN_GROUP
for spec in SPECS:
    if spec['fusion'] not in ('crossgate', 'concat'):
        raise ValueError(f"Unknown fusion: {spec['fusion']}")
    if len(spec['view_indices']) != spec['n_2d']:
        raise ValueError(f"{spec['code']}: view_indices/n_2d mismatch")
    if spec['fusion'] == 'crossgate' and not spec['use_3d']:
        raise ValueError(f"{spec['code']}: CrossGate requires the 3D branch")
selected = [(spec, seed, f"{spec['code']}_s{seed}") for spec in SPECS for seed in SEEDS if not RUN_TARGET or RUN_TARGET == f"{spec['code']}_s{seed}"]
if not selected:
    raise ValueError('RUN_TARGET does not match SPECS/SEEDS')
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError('Drive must be a verified mounted filesystem, not a local directory')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = ft.Artifacts(LOCAL_ROOT, None if SMOKE else DRIVE_DIR, smoke=SMOKE)
DATA_STORAGE = ft.Artifacts(DATA_ROOT, None if SMOKE else DRIVE_DIR / 'data', smoke=SMOKE)
print('Device:', DEVICE, '| specs:', [spec['code'] for spec in SPECS], '| seeds:', SEEDS, '| runs:', len(selected))

## Data
D1 downloads only the declared splits from HF with `HF_TOKEN` (`allow_patterns`); D2 verifies the raw
storage; D3 builds/caches the 2D views + depth-axis once per split; D4 defines the dataset factory used by
the training loop.

In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in zip(SPLITS, (10, 6, 6)):
        np.save(DATA_ROOT / f'{split}_volumes.npy', rng.integers(0, 255, (n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8))
        np.save(DATA_ROOT / f'{split}_labels.npy', np.arange(n, dtype=np.int64) % 2)
    print('[data] smoke synthetic arrays ready at', DATA_ROOT)
else:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Authenticated HF download requires HF_TOKEN in .env/environment or Colab Secrets')
    print('[data] repo', HF_DATA_REPO, '| patterns', len(HF_DATA_PATTERNS))
    if not all((DATA_ROOT / name).is_file() for name in HF_DATA_PATTERNS):
        print('[data] downloading declared splits (authenticated)...')
        snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=token, allow_patterns=HF_DATA_PATTERNS)
        print('[data] download complete')
    else:
        print('[data] cache hit; no download needed')

In [ ]:
for split in SPLITS:
    volumes = np.load(DATA_ROOT / f'{split}_volumes.npy', mmap_mode='r')
    labels = np.load(DATA_ROOT / f'{split}_labels.npy')
    if not SMOKE and tuple(volumes.shape[-3:]) != (STORE_RES, STORE_RES, STORE_RES):
        raise ValueError(f'Real training requires STORE_RES-cubed storage, got {volumes.shape[-3:]}')
    if len(volumes) != len(labels):
        raise ValueError(f'{split}: volume/label count mismatch')
    print(f'[storage] {split}: {tuple(volumes.shape)} | labels {len(labels)} | pos {int(np.asarray(labels).sum())}')

In [ ]:
for split in SPLITS:
    views_path, depth_path = ft.build_views(DATA_ROOT / f'{split}_volumes.npy', res2d=RES2D)
    print(f'[views] {split}: {views_path.name} + {depth_path.name}')
for split in SPLITS:
    for path in DATA_ROOT.glob(f'{split}_volumes_*{RES2D}*'):
        if '.partial.' not in path.name:
            DATA_STORAGE.sync(path)
print('[cache] view/depth caches synced')

In [ ]:
def make_datasets(ds, seed):
    if ds != 'raw':
        raise ValueError('Controls notebook trains on raw storage only')
    datasets = [ft.FinalDataset(DATA_ROOT / f'{s}_volumes.npy', DATA_ROOT / f'{s}_labels.npy', res3d=RES3D, res2d=RES2D, seed=seed, train=s == 'Training') for s in SPLITS]
    for split, dataset in zip(SPLITS, datasets):
        print(f'[dataset] {split}: n={len(dataset)} pos={int(np.asarray(dataset.labels).sum())} | res3d={RES3D} | views {RES2D}px | {dataset.source.name}')
    return datasets

## Train, evaluate, persist and explain
Factories keep each cell small: model init (per spec), batch/config resolution (VRAM probe), val/test
callbacks, and report/checkpoint finalization. Each spec x seed is a separate run (`ctrl_<code>_s<seed>`),
resume-safe, with the same metrics cadence and calibrated `train/val/test` report as the canonical notebook.

In [ ]:
def make_model(spec, tag_resume, artifacts):
    model_path = artifacts.local / 'best_weights.pt'
    if tag_resume or not model_path.is_file():
        model_path = None
    model = (ft.SmokeModel() if SMOKE else cm.ControlsModel(n_2d=spec['n_2d'], D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=not (tag_resume or model_path is not None), view_indices=spec['view_indices'], fusion=spec['fusion'], use_3d=spec['use_3d'], gate_fixed=spec['gate_fixed'])).to(DEVICE)
    if model_path is not None:
        ft.load_weights(model, model_path)
        print('[model] loaded weights from', model_path)
    else:
        print(f"[model] new model for {spec['code']} | fusion={spec['fusion']} views={spec['view_indices']} use_3d={spec['use_3d']} gate_fixed={spec['gate_fixed']}")
    return model

In [ ]:
def resolve_batch(tag, tr, spec):
    if RESUME:
        saved = ft.saved_config(LOCAL_ROOT / tag, DRIVE_DIR / tag)
        if saved:
            print('[batch] resume reuse bs/accum', int(saved['batch_size']), int(saved['grad_accum']))
            return int(saved['batch_size']), int(saved['grad_accum']), None
    if AUTO_BATCH and DEVICE.type == 'cuda':
        probe = ft.find_batch_size(lambda: ft.SmokeModel() if SMOKE else cm.ControlsModel(n_2d=spec['n_2d'], D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=False, view_indices=spec['view_indices'], fusion=spec['fusion'], use_3d=spec['use_3d'], gate_fixed=spec['gate_fixed']), tr, device=DEVICE, start=BS, target_gb=TARGET_VRAM_GB, num_workers=NUM_WORKERS)
        bs = int(probe['batch_size'])
        print('[batch] probe result:', probe)
        return bs, max(1, round(EFFECTIVE_BATCH / bs)), probe
    return BS, GRAD_ACCUM, None

def make_config(spec, seed, tr, va, te, bs, accum):
    ytr = tr.labels
    if set(np.unique(ytr)) != {0, 1}:
        raise ValueError('Training requires both binary classes')
    weights = [len(ytr) / (2 * int((ytr == c).sum())) for c in (0, 1)]
    return dict(dataset='raw', seed=seed, spec=spec['code'], fusion=spec['fusion'], view_indices=list(spec['view_indices']), use_3d=spec['use_3d'], gate_fixed=spec['gate_fixed'], epochs=EPOCHS, batch_size=bs, grad_accum=accum, lr=LR, weight_decay=WD, patience=PATIENCE, checkpoint_steps=CHECKPOINT_EVERY_STEPS, class_weights=weights, res3d=RES3D, res2d=RES2D, store_res=STORE_RES, n2d=spec['n_2d'], latent=D_LATENT, enc2d=ENC2D, enc3d_features=list(ENC3D_FEATURES), smoke=SMOKE, torch_version=str(torch.__version__), device=str(DEVICE), data=[ft.data_identity(p) for d in (tr, va, te) for p in (d.source, d.label_path)])

def log_batch_probe(run, probe, bs, accum):
    if not probe:
        return
    table = wandb.Table(columns=['batch_size', 'peak_gb', 'ok'])
    for trial in probe['trials']:
        table.add_data(trial['batch_size'], trial['peak_gb'], trial['ok'])
    run.log({'report/batch_probe_table': table})
    run.summary.update({'batch/size': bs, 'batch/accum': accum, 'batch/peak_gb': probe['peak_gb'], 'batch/target_gb': TARGET_VRAM_GB})

In [ ]:
METRIC_KEYS = ('loss', 'acc', 'balanced_acc', 'precision', 'recall', 'specificity', 'npv', 'f1', 'mcc', 'kappa', 'youden', 'auc_roc', 'auc_pr', 'ece', 'logloss', 'brier')

def metric_line(metrics):
    return ' '.join(f'{key}={metrics[key]:.4f}' for key in METRIC_KEYS if key in metrics)

def make_val_eval(va, bs, progress):
    def evaluate(model):
        p, y, logits = ft.predict(model, va, bs, num_workers=NUM_WORKERS)
        metrics = {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        epoch = progress['trainer'].epoch + 1 if progress['trainer'] is not None else 0
        print(f'[val  ] epoch {epoch}: {metric_line(metrics)}', flush=True)
        return metrics
    return evaluate

def make_test_eval(te, bs, progress):
    def evaluate_test(model):
        p, y, logits = ft.predict(model, te, bs, num_workers=NUM_WORKERS)
        metrics = {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        epoch = progress['trainer'].epoch if progress['trainer'] is not None else 0
        print(f'[test ] epoch {epoch}: {metric_line(metrics)}', flush=True)
        return metrics
    return evaluate_test


In [ ]:
def finish_tag(model, trainer, tr, va, te, artifacts, run, tag, seed, started, bs):
    res, probs, labels = ft.calibrated_report(model, va, te, bs, smoke=SMOKE, train=tr, num_workers=NUM_WORKERS)
    res.update(tag=tag, seed=seed, hist=trainer.history, minutes=round((time.time() - started) / 60, 2))
    print(f'[eval] test AUC={res["test"]["auc_roc"]:.4f} F1={res["test"]["f1"]:.4f} | val AUC={res["val"]["auc_roc"]:.4f}')
    ft.log_report(run, res)
    params = sum(p.numel() for p in model.parameters())
    run.summary.update({'threshold': res['threshold'], 'temperature': res['temperature'], 'params': params, 'minutes': res['minutes']})
    weights_path = artifacts.save(ft.cpu_state(model), 'best_weights.pt')
    print('[checkpoint] saved', weights_path.name)
    ft.save_report(res, probs, labels, artifacts, run)
    if RUN_XAI:
        ft.save_xai(model, va, artifacts, run, smoke=SMOKE)
    return dict(res=res, weights_path=str(weights_path), params=params, test_probs=probs.tolist(), test_labels=labels.tolist())

In [ ]:
RESULTS = {}
ACTIVE_MODEL = ACTIVE_TRAINER = WANDB_RUN = None
LAST = {}
for spec, seed, tag in selected:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te = make_datasets('raw', seed)
    bs, accum, batch_probe = resolve_batch(tag, tr, spec)
    config = make_config(spec, seed, tr, va, te, bs, accum)
    artifacts = ft.Artifacts(LOCAL_ROOT / tag, None if SMOKE else DRIVE_DIR / tag, smoke=SMOKE)
    status = ft.run_status(artifacts, config, resume=RESUME, extend_epochs=EXTEND_EPOCHS)
    config = status['config']
    print(f'[run] {tag}: status={status["status"]} epochs={config["epochs"]}')
    LAST = {'tag': tag, 'artifacts': artifacts, 'config': config}
    if status['status'] == 'complete':
        RESULTS[tag] = {'res': status['result']}
        continue
    tag_resume = status['status'] == 'resume'
    WANDB_RUN = ft.init_wandb('ctrl_' + tag, config, artifacts, resume=status['status'] != 'new', smoke=SMOKE)
    log_batch_probe(WANDB_RUN, batch_probe, bs, accum)
    stopped, started = False, time.time()
    progress = {'trainer': None}
    try:
        ACTIVE_MODEL = make_model(spec, tag_resume, artifacts)
        ACTIVE_TRAINER = ft.Trainer(ACTIVE_MODEL, tr, config, artifacts, WANDB_RUN, resume=tag_resume, num_workers=NUM_WORKERS)
        progress['trainer'] = ACTIVE_TRAINER
        print(f'[train] start {tag}: epochs={config["epochs"]} bs={bs} accum={accum} eff={bs * accum} workers={NUM_WORKERS} cuda={DEVICE.type == "cuda"}')
        timer = {'epoch': 0, 'started': time.time()}
        def epoch_timer(trainer):
            if trainer.epoch != timer['epoch']:
                now = time.time()
                seconds = now - timer['started']
                completed = trainer.epoch
                timer['epoch'], timer['started'] = completed, now
                trainer.run.log({'progress/step': trainer.step, 'train/epoch_seconds': seconds})
                print(f'[train] epoch {completed} done in {seconds:.1f}s', flush=True)
        stopped = not ACTIVE_TRAINER.fit(make_val_eval(va, bs, progress), test_evaluate=make_test_eval(te, bs, progress), boundary_hook=epoch_timer)
        if ACTIVE_TRAINER.epoch > timer['epoch']:
            seconds = time.time() - timer['started']
            ACTIVE_TRAINER.run.log({'progress/step': ACTIVE_TRAINER.step, 'train/epoch_seconds': seconds})
            print(f'[train] epoch {ACTIVE_TRAINER.epoch} done in {seconds:.1f}s', flush=True)
        print('[train] fit finished | stopped =', stopped)
        if not stopped:
            ACTIVE_MODEL.load_state_dict(ACTIVE_TRAINER.best_state)
            RESULTS[tag] = finish_tag(ACTIVE_MODEL, ACTIVE_TRAINER, tr, va, te, artifacts, WANDB_RUN, tag, seed, started, bs)
    except BaseException:
        WANDB_RUN.finish(exit_code=1)
        raise
    if stopped:
        WANDB_RUN.summary['stopped_safely'] = True
        WANDB_RUN.finish(exit_code=0)
        print('Stopped safely. Set RESUME=True and RUN_TARGET to', tag)
        break
    ft.complete_run(artifacts, config, WANDB_RUN.id)
    ACTIVE_MODEL.cpu()
    ACTIVE_TRAINER = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Completed:', list(RESULTS))

## Aggregate results (mean +/- std across seeds)
Reads every finished run from local + Drive, builds the per-spec aggregate table (mean/std/min/max for all
val/test metrics), writes `controls_96_summary.csv/json`, logs the table to W&B and syncs Drive.

In [ ]:
rows = ft.publish_summary(STORAGE)
summary = {}
if rows:
    for row in rows:
        code = row['tag'].split('_s')[0]
        summary.setdefault(code, []).append(row)
if summary:
    metric_keys = sorted({key for row in rows for key in row if key.startswith(('val_', 'test_')) and key not in ('val_n', 'test_n')})
    agg = {}
    for code, items in summary.items():
        agg[code] = {'n_seeds': len(items), 'seeds': sorted(int(item['seed']) for item in items)}
        for key in metric_keys:
            values = np.asarray([item[key] for item in items], dtype=float)
            agg[code][key] = {'mean': float(values.mean()), 'std': float(values.std(ddof=1)) if len(values) > 1 else 0.0, 'min': float(values.min()), 'max': float(values.max())}
    order = [spec['code'] for spec in SPECS if spec['code'] in agg]
    lines = ['spec,n_seeds,' + ','.join(f'{key}_mean,{key}_std' for key in metric_keys)]
    for code in order:
        item = agg[code]
        lines.append(code + ',' + str(item['n_seeds']) + ',' + ','.join(f"{item[key]['mean']:.4f},{item[key]['std']:.4f}" for key in metric_keys))
    csv_path = STORAGE.local / 'controls_96_summary.csv'
    csv_path.write_text('\n'.join(lines))
    STORAGE.sync(csv_path)
    STORAGE.save(agg, 'controls_96_summary.pt')
    print('[summary] specs:', order)
    for code in order:
        item = agg[code]
        print(f"[summary] {code}: n={item['n_seeds']} test_auc_roc={item['test_auc_roc']['mean']:.4f}+-{item['test_auc_roc']['std']:.4f} test_f1={item['test_f1']['mean']:.4f}+-{item['test_f1']['std']:.4f}")
    summary_run = wandb.init(project='glaucoma-thesis', name='controls_96_summary_' + time.strftime('%Y%m%d_%H%M%S'), config={'run_group': RUN_GROUP, 'specs': order}, reinit=True, mode='offline' if SMOKE else 'online')
    table = wandb.Table(columns=['spec', 'n_seeds', 'test_auc_roc_mean', 'test_auc_roc_std', 'test_f1_mean', 'test_f1_std'])
    for code in order:
        item = agg[code]
        table.add_data(code, item['n_seeds'], item['test_auc_roc']['mean'], item['test_auc_roc']['std'], item['test_f1']['mean'], item['test_f1']['std'])
    summary_run.log({'report/controls_summary': table})
    summary_run.finish(exit_code=0)
else:
    print('No completed runs; no aggregate table yet.')
print('Smoke verified only local/offline behavior.' if SMOKE else 'Aggregates synced to the verified Drive mount.')

## Outputs and limitations
- Runs: `outputs/crossgate_controls_96/<RUN_GROUP>/<spec>_s<seed>/` with `metrics.json`, `test_predictions.pt`
  and `best_weights.pt`; W&B runs `ctrl_<spec>_s<seed>` in project `glaucoma-thesis`.
- Aggregate: `controls_96_summary.csv` / `controls_96_summary.pt` + `report/controls_summary` table in W&B.
- Every spec is trained from scratch on the same split/seed/protocol; best checkpoint by validation AUC,
  early stop with the same patience. Single-spec differences near noise should be read with the per-seed std
  and the calibrated CIs (not accuracy alone).
- D1/D2 (Bilateral 5-epoch fine-tunes from the same raw checkpoint) are run from the final notebook; merge
  their metrics into the same comparison table when they finish.